> **Note (Nov 2026):** the recorded outputs below are from the 27 Jul 2026 run of the *first* version of the pipeline. That run only had data for 5 of the 8 clause types and compared a token-level val F1 to a span-level test F1 — see the README's *Results* section. Re-run all cells to get numbers for the current code.

# CUAD Clause Extractor — Training Pipeline (Kaggle)

**Before running:** In the right sidebar, set **Accelerator → GPU T4 x2** (or P100).  
Also set **Internet → On** (needed to clone the repo and download CUAD from HuggingFace).

**Fixed in this version** (after debugging a NaN-loss / zero-F1 issue over several runs):
- `transformers` pinned to `4.44.2` in `requirements.txt` — an unpinned `>=4.40.0` was resolving to a much newer release that combined with DeBERTa-v3 to explore an unstable region
- `AdamW(eps=1e-6)` instead of the PyTorch default `1e-8` — the real fix for the NaN-after-one-step pattern; DeBERTa-v3's gradient-disentangled embedding sharing produces very small early gradients that make the default epsilon spike the adaptive step size
- Class-weighted loss — the model was collapsing to predicting all-"O" (confirmed via diagnostics: 0/14946 non-O tokens predicted across 3 epochs) because clause spans are a small minority of tokens; inverse-frequency weights fix this
- GPU memory freed between hyperparameter configs (fixes an OOM crash partway through the sweep)
- `max_length=384` for tokenization (safety margin below the 512 position-embedding limit)
- A checkpoint is always saved even if val F1 never improves past 0, so `quantize.py`/`evaluate.py` never crash on a missing model directory
- Cell 2 does a full fresh clone every run, which also clears any stale `mlflow.db`/`mlruns` state that previously caused a "readonly database" crash

Steps: clone repo -> prepare data -> hyperparameter search -> quantize -> evaluate -> calibration


In [1]:
# ── 1. Check GPU ──────────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('No GPU found. In the right sidebar: Accelerator -> GPU T4 x2')
print(result.stdout[:600])

Mon Jul 27 08:33:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|===============


In [2]:
# ── 2. Clone repo (fresh every run — also clears any stale mlflow state) ────
%cd /kaggle/working
!rm -rf cuad-extractor
!git clone https://github.com/Pranjal-agl/cuad-extractor.git
%cd cuad-extractor
!git log --oneline -3

/kaggle/working
Cloning into 'cuad-extractor'...
remote: Enumerating objects: 90, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 90 (delta 50), reused 69 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (90/90), 37.28 KiB | 5.32 MiB/s, done.
Resolving deltas: 100% (50/50), done.
/kaggle/working/cuad-extractor
e028980 (HEAD -> main, origin/main, origin/HEAD) sigh
63dcec6 27jul
612a6f1 x3.1


In [3]:
# ── 3. Install dependencies ──────────────────────────────────────────────────
# transformers stays pinned (verified working: 4.44.2).
# bitsandbytes is force-upgraded to >=0.45.2 -- Kaggle's preinstalled
# bitsandbytes is older and crashes with "No module named 'triton.ops'"
# against the newer triton shipped in this image (triton removed that
# module; bitsandbytes<0.45.2 still imports it unconditionally). A loose
# >=0.43.0 bound in requirements.txt does NOT trigger an upgrade if the
# already-installed version satisfies it, so --upgrade is required here.
!pip install -q "transformers==4.44.2"
!pip install -q --upgrade "bitsandbytes>=0.45.2"
!pip install -q mlflow accelerate peft evaluate python-dotenv
!pip install -q -r requirements.txt
import transformers, bitsandbytes
print('transformers version:', transformers.__version__)
print('bitsandbytes version:', bitsandbytes.__version__)
assert transformers.__version__ == '4.44.2', 'transformers was upgraded by a later install step -- check requirements.txt pin'
import re
bnb_major, bnb_minor = (int(x) for x in bitsandbytes.__version__.split('.')[:2])
assert (bnb_major, bnb_minor) >= (0, 45), f'bitsandbytes {bitsandbytes.__version__} may still hit the triton.ops import error -- expected >=0.45.2'
print('Done -- versions OK')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 83.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 99.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 50.2 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 101.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 94.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━

In [4]:
# ── 4. Prepare data ──────────────────────────────────────────────────────────
# Note: this pulls CUAD from HuggingFace unauthenticated by default, which can
# occasionally return rate-limited/partial data. If you see the example count
# vary noticeably between runs, add an HF_TOKEN via Kaggle Secrets and
# `os.environ['HF_TOKEN'] = ...` before this cell for consistent pulls.
!python -m src.training.prepare_data

Loading clause labels...
README.md: 6.87kB [00:00, 22.1MB/s]
data/train-00000-of-00001.parquet: 100%|███| 1.80M/1.80M [00:00<00:00, 2.13MB/s]
Generating train split: 100%|██| 13155/13155 [00:00<00:00, 144217.60 examples/s]
  13155 rows, columns: ['file_name', 'clause', 'pages', 'class_id', 'label', 'start_at', 'end_at']
Loading full contract text...
README.md: 4.40kB [00:00, 1.60MB/s]
data/train-00000-of-00001.parquet: 100%|█████| 139M/139M [00:02<00:00, 57.4MB/s]
Generating train split: 100%|████████| 509/509 [00:00<00:00, 1131.29 examples/s]
  509 contracts
  Skipped 11413 rows (not in target types or missing text)
Found 458 contracts, 1742 examples
Split: 1240 train / 233 val / 269 test
Contamination check passed. train=320, val=68, test=70 contracts
  train: 1240 examples (1240 positive) -> /kaggle/working/cuad-extractor/data/train.jsonl
  val: 233 examples (233 positive) -> /kaggle/working/cuad-extractor/data/val.jsonl
  test: 269 examples (269 positive) -> /kaggle/working/cuad-ex

In [5]:
# ── 5. Verify splits ─────────────────────────────────────────────────────────
import json
from pathlib import Path
from collections import Counter

for split in ['train', 'val', 'test']:
    path = Path(f'data/{split}.jsonl')
    examples = [json.loads(l) for l in path.read_text().splitlines() if l.strip()]
    clause_counts = Counter(e['clause'] for e in examples)
    pos = sum(1 for e in examples if e['has_answer'])
    print(f'\n{split}: {len(examples)} total, {pos} positive')
    for clause, count in sorted(clause_counts.items()):
        p = sum(1 for e in examples if e['clause'] == clause and e['has_answer'])
        print(f'  {clause:<35} {p}/{count}')


train: 1240 total, 1240 positive
  Cap On Liability                    436/436
  Governing Law                       319/319
  Non-Compete                         179/179
  Renewal Term                        142/142
  Termination For Convenience         164/164

val: 233 total, 233 positive
  Cap On Liability                    91/91
  Governing Law                       69/69
  Non-Compete                         22/22
  Renewal Term                        27/27
  Termination For Convenience         24/24

test: 269 total, 269 positive
  Cap On Liability                    109/109
  Governing Law                       67/67
  Non-Compete                         27/27
  Renewal Term                        30/30
  Termination For Convenience         36/36


In [6]:
# ── 6. Train with hyperparameter search ──────────────────────────────────────
# 6 configs, best selected by val F1. Kaggle sessions run up to 12h/session
# and 30h/week GPU quota, more headroom than Colab free tier.
# If you hit a wall, reduce HPARAM_GRID in src/training/train.py to 2-3 configs.
#
# Watch for these lines in the output:
#   "Class weights (...)"        -> confirms the imbalance-fix is active
#   "[LABEL-DIST] predicted..."  -> should show non-zero non-O predictions
#                                    once the model starts learning (not 0
#                                    like in the earlier broken runs)
#   "val_macro_f1=..."            -> should be > 0 by epoch 1-2
!python -m src.training.train

Test set lock verified.
Train: 1240 | Val: 233
tokenizer_config.json: 100%|██████████████████| 52.0/52.0 [00:00<00:00, 257kB/s]
config.json: 100%|█████████████████████████████| 579/579 [00:00<00:00, 3.49MB/s]
spm.model: 100%|███████████████████████████| 2.46M/2.46M [00:01<00:00, 2.41MB/s]
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:551: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produc

In [7]:
# ── 7. Check best config ─────────────────────────────────────────────────────
import json
best = json.loads(open('models/best_config.json').read())
print('Best config:')
print(f"  Run:     {best['run_name']}")
print(f"  Val F1:  {best['val_f1']:.4f}")
print(f"  Config:  {best['config']}")
print(f"  Model:   {best['model_path']}")

Best config:
  Run:     run_3_lr2e-05_bs8_ep3
  Val F1:  0.4318
  Config:  {'lr': 2e-05, 'batch_size': 8, 'epochs': 3, 'warmup_ratio': 0.1}
  Model:   /kaggle/working/cuad-extractor/models/run_3_lr2e-05_bs8_ep3


In [8]:
# ── 8. Quantize (FP32 -> 4-bit NF4) ─────────────────────────────────────────
!python -m src.training.quantize

Loading FP32 model...
FP32: F1=0.4255  p99=29.21ms  size=711.9MB
Loading 4-bit quantized model...
`low_cpu_mem_usage` was None, now set to True since model is quantized.
  [WARNING] 4-bit quantization failed (known upstream bug): `.to` is not supported for `4-bit` or `8-bit` bitsandbytes models. Please use the model as it is, since the model has already been set to the correct devices and casted to the correct `dtype`.
  Continuing with FP32-only results.

Report written -> /kaggle/working/cuad-extractor/experiments/quantization_report.json

  QUANTIZATION SUMMARY
  FP32   F1=0.4255  p99=29.21ms  711.9MB
  p99 target (150ms) met by FP32: YES


In [9]:
# ── 9. Read quantization report ──────────────────────────────────────────────
import json
report = json.loads(open('experiments/quantization_report.json').read())
fp32 = report['fp32']
print(f"FP32 : F1={fp32['val_macro_f1']}  CPU p99={fp32['latency_cpu']['p99_ms']}ms  "
      f"GPU p99={fp32.get('latency_gpu', {}).get('p99_ms')}ms  {fp32['model_size_mb']}MB")
for name in ('int8_dynamic_cpu', 'nf4_gpu'):
    r = report.get(name, {})
    if 'val_macro_f1' in r:
        lat = r.get('latency_cpu') or r.get('latency_gpu')
        print(f"{name}: F1={r['val_macro_f1']}  drop={r['f1_drop']}  p99={lat['p99_ms']}ms  {r['model_size_mb']}MB  "
              f"speedup={r.get('cpu_speedup_p99') or r.get('gpu_speedup_p99')}x")
    else:
        print(f"{name}: {r}")


FP32:  F1=0.4255  p99=29.21ms  711.9MB
4-bit: skipped (known upstream transformers/bitsandbytes dispatch_model bug -- see quantize.py comments)
p99 target (150ms) met: True


In [10]:
# ── 10. Evaluate on test set (run ONCE after training is done) ───────────────
!python -m src.eval.evaluate --split test

Test set lock verified.
Loading FP32 model from /kaggle/working/cuad-extractor/models/run_3_lr2e-05_bs8_ep3...
Evaluating on 269 test examples...
  [test] 50/269
  [test] 100/269
  [test] 150/269
  [test] 200/269
  [test] 250/269

Running adversarial eval...
  [FAIL] truncated_context: f1=0.468
  [PASS] no_clause_present: f1=1.000
  [FAIL] multiple_spans: f1=0.338
  [PASS] boilerplate_noise: f1=1.000

Results written -> /kaggle/working/cuad-extractor/experiments/eval_test_fp32.json

  EVAL RESULTS (test, FP32)
  Macro F1 : 0.7747
  Val F1   : 0.4318  (overfit gap: -0.3429)

  Per-clause F1:
    Governing Law                       0.9725  CI=[0.920, 0.997]  n=67
    Termination For Convenience         0.6548  CI=[0.589, 0.863]  n=36
    Renewal Term                        0.7636  CI=[0.744, 0.965]  n=30
    Non-Compete                         0.7538  CI=[0.719, 0.962]  n=27
    Cap On Liability                    0.7290  CI=[0.693, 0.847]  n=109

  Latency: p50=21.84ms  p95=23.36ms  p99

In [11]:
# ── 11. Read eval results ────────────────────────────────────────────────────
import json, glob
results_files = glob.glob('experiments/eval_test_*.json')
if results_files:
    results = json.loads(open(sorted(results_files)[-1]).read())
    print(f"Macro F1:     {results['macro_f1']:.4f}")
    print(f"Val F1:       {results.get('val_macro_f1', 'N/A')}")
    print(f"Test - val:   {results.get('generalization_gap', results.get('overfit_gap', 'N/A'))}")
    print(f"p99 latency:  {results['latency']['p99_ms']}ms")
    print(f"p99 target met: {results['latency']['target_met']}")
    print(f"Adversarial pass rate: {results['adversarial']['pass_rate']:.0%}")
    print()
    print('Per-clause F1:')
    for clause, m in results['per_clause'].items():
        if m['f1'] is not None:
            ci = m['ci_95']
            print(f"  {clause:<35} F1={m['f1']:.4f}  CI=[{ci[0]:.3f}, {ci[1]:.3f}]")

Macro F1:     0.7747
Val F1:       0.4318
Overfit gap:  -0.3429
p99 latency:  29.12ms
p99 target met: True
Adversarial pass rate: 50%

Per-clause F1:
  Governing Law                       F1=0.9725  CI=[0.920, 0.997]
  Termination For Convenience         F1=0.6548  CI=[0.589, 0.863]
  Renewal Term                        F1=0.7636  CI=[0.744, 0.965]
  Non-Compete                         F1=0.7538  CI=[0.719, 0.962]
  Cap On Liability                    F1=0.7290  CI=[0.693, 0.847]


In [12]:
# ── 12. Calibration proxy report ─────────────────────────────────────────────
!python -m src.eval.calibration


  Calibration Proxy Report
  Source: eval_test_fp32.json

  Clause                                   F1    Prec     Rec   n_pos     n
  Governing Law                         0.973   0.966   0.987      67    67
  Termination For Convenience           0.655   0.672   0.780      36    36
  Renewal Term                          0.764   0.716   0.919      30    30
  Non-Compete                           0.754   0.654   0.961      27    27
  Cap On Liability                      0.729   0.692   0.856     109   109



In [13]:
# ── 13. Copy results to /kaggle/working for download ─────────────────────────
# Kaggle auto-persists everything under /kaggle/working — after the run,
# use the notebook's "Output" tab on the right to download files, or download
# the zip directly below.
!zip -r /kaggle/working/experiments_results.zip experiments/ models/best_config.json
print('Saved to /kaggle/working/experiments_results.zip')
print('Download via the Output panel on the right side of the Kaggle notebook.')

  adding: experiments/ (stored 0%)
  adding: experiments/quantization_report.json (deflated 56%)
  adding: experiments/load_test_report.json (deflated 40%)
  adding: experiments/eval_test_fp32.json (deflated 69%)
  adding: models/best_config.json (deflated 34%)
Saved to /kaggle/working/experiments_results.zip
Download via the Output panel on the right side of the Kaggle notebook.
